In [1]:
import pandas as pd
import numpy as np
import os
import glob

# 1. LOAD PROCESSED HPI TARGET VARIABLE
print("Loading processed HPI dataset...")
df_hpi = pd.read_csv('../data/processed/kansas_hpi_processed.csv')

# Ensure standard year and quarter types for smooth merging
df_hpi['year'] = df_hpi['year'].astype(int)
df_hpi['quarter'] = df_hpi['quarter'].astype(int)

print(f"HPI data loaded successfully. Shape: {df_hpi.shape}")
display(df_hpi.head())

Loading processed HPI dataset...
HPI data loaded successfully. Shape: (912, 7)


,metro_name,msa_code,year,quarter,index_value,standard_error,quarterly_growth
0,"Kansas City, MO-KS",28140,1976,2,45.25,( 0.54),0.689809
1,"Kansas City, MO-KS",28140,1976,3,46.30,( 0.48),2.320442
2,"Kansas City, MO-KS",28140,1976,4,46.19,( 0.50),-0.237581
3,"Kansas City, MO-KS",28140,1977,1,47.15,( 0.57),2.078372
4,"Kansas City, MO-KS",28140,1977,2,49.72,( 0.45),5.450689


In [3]:
# 2. PROCESS MORTGAGE RATES
mortgage_path = '../data/raw/MORTGAGE30US.csv' 

print("Processing Mortgage Rates...")
df_mortgage = pd.read_csv(mortgage_path)

# Assuming FRED format (Date column and rate column)
# Convert date to datetime, extract year and quarter, then average weekly/monthly rates to quarterly
df_mortgage['date'] = pd.to_datetime(df_mortgage.iloc[:, 0]) # Adjust column index if needed
df_mortgage['year'] = df_mortgage['date'].dt.year
df_mortgage['quarter'] = df_mortgage['date'].dt.quarter

rate_col_name = df_mortgage.columns[1] # Assumes value is in the 2nd column
df_mortgage_q = df_mortgage.groupby(['year', 'quarter'])[rate_col_name].mean().reset_index()
df_mortgage_q.rename(columns={rate_col_name: 'mortgage_rate'}, inplace=True)

display(df_mortgage_q.head())

Processing Mortgage Rates...


,year,quarter,mortgage_rate
0,1971,2,7.413077
1,1971,3,7.658462
2,1971,4,7.552857
3,1972,1,7.349231
4,1972,2,7.346923


In [5]:
# 3. PROCESS MEDIAN HOUSEHOLD INCOME (Kansas Statewide)
income_path = '../data/raw/KSMedianIncome_MEHOINUSKSA646N.csv'

print("Processing Median Household Income...")
df_income = pd.read_csv(income_path)

# Assign clear column names based on your file structure
df_income.columns = ['date', 'median_income']

# Convert date to datetime, extract year
df_income['date'] = pd.to_datetime(df_income['date'])
df_income['year'] = df_income['date'].dt.year

# Clean and convert income values to numeric
df_income['median_income'] = pd.to_numeric(df_income['median_income'], errors='coerce')
df_income = df_income[['year', 'median_income']].dropna()

display(df_income.head())

Processing Median Household Income...


,year,median_income
0,1984,24630
1,1985,22790
2,1986,23930
3,1987,25580
4,1988,25570


In [7]:
# 4. PROCESS UNEMPLOYMENT FOLDER
unemployment_dir = '../data/raw/Unemployment/'
print("Processing Unemployment files from folder...")

unemployment_dfs = []

metro_name_mapping = {
    'KANS129URN.csv': 'Kansas City, MO-KS',
    'WICH620URN.csv': 'Wichita, KS',
    'LAWR920URN.csv': 'Lawrence, KS',
    'LAUMT203174000000003.csv': 'Manhattan, KS',
    'TOPE820URN.csv': 'Topeka, KS'
}

for file_path in glob.glob(os.path.join(unemployment_dir, '*.csv')):
    file_name = os.path.basename(file_path)
    
    # Load individual metro unemployment file
    temp_df = pd.read_csv(file_path)
    
    # FRED format check: 2 columns -> date and unemployment rate value
    temp_df.columns = ['date', 'unemployment_rate']
    
    # Convert date, extract year and quarter
    temp_df['date'] = pd.to_datetime(temp_df['date'], errors='coerce')
    temp_df['year'] = temp_df['date'].dt.year
    temp_df['quarter'] = temp_df['date'].dt.quarter
    
    # Clean unemployment rate values (handle '.' or missing strings if present)
    temp_df['unemployment_rate'] = pd.to_numeric(temp_df['unemployment_rate'], errors='coerce')
    
    # Aggregate monthly rates to quarterly means
    temp_df_q = temp_df.groupby(['year', 'quarter'])['unemployment_rate'].mean().reset_index()
    
    # Assign the corresponding standardized metro name
    # Fallback to file name if not explicitly in dictionary
    assigned_metro = metro_name_mapping.get(file_name, file_name.replace('.csv', ''))
    temp_df_q['metro_name'] = assigned_metro
    
    unemployment_dfs.append(temp_df_q)

# Combine all regional unemployment dataframes into one master frame
if unemployment_dfs:
    df_unemp_master = pd.concat(unemployment_dfs, ignore_index=True)
    print(f"Successfully processed {len(unemployment_dfs)} unemployment regional files!")
    display(df_unemp_master.head())
else:
    print("No unemployment CSV files found in the folder. Check directory path.")

Processing Unemployment files from folder...
Successfully processed 6 unemployment regional files!


,year,quarter,unemployment_rate,metro_name
0,1990,1,5.133333,"Kansas City, MO-KS"
1,1990,2,4.433333,"Kansas City, MO-KS"
2,1990,3,5.033333,"Kansas City, MO-KS"
3,1990,4,4.800000,"Kansas City, MO-KS"
4,1991,1,6.400000,"Kansas City, MO-KS"


In [8]:
# 5. PROCESS BUILDING PERMITS FOLDER
permits_dir = '../data/raw/Building Permits/'
print("Processing Building Permits files from folder...")

permits_dfs = []

permits_name_mapping = {
    'KCKMO_KANS129BPPRIV.csv': 'Kansas City, MO-KS',
    'Wichita_BPPRIV020173.csv': 'Wichita, KS',
    'Lawrence_BPPRIV020045.csv': 'Lawrence, KS',
    'Manhattan_BPPRIV020061.csv': 'Manhattan, KS',
    'Topeka_BPPRIV020177.csv': 'Topeka, KS'
}

for file_path in glob.glob(os.path.join(permits_dir, '*.csv')):
    file_name = os.path.basename(file_path)
    
    # Load individual metro building permits file
    temp_df = pd.read_csv(file_path)
    
    # FRED format check: 2 columns -> date and permit values
    temp_df.columns = ['date', 'building_permits']
    
    # Convert date, extract year and quarter
    temp_df['date'] = pd.to_datetime(temp_df['date'], errors='coerce')
    temp_df['year'] = temp_df['date'].dt.year
    temp_df['quarter'] = temp_df['date'].dt.quarter
    
    # Clean and convert permit counts to numeric (dropping any missing values)
    temp_df['building_permits'] = pd.to_numeric(temp_df['building_permits'], errors='coerce')
    
    # Aggregate monthly permit counts to quarterly SUMS (since permits accumulate over time)
    temp_df_q = temp_df.groupby(['year', 'quarter'])['building_permits'].sum().reset_index()
    
    # Assign the corresponding standardized metro name
    assigned_metro = permits_name_mapping.get(file_name, file_name.replace('.csv', ''))
    temp_df_q['metro_name'] = assigned_metro
    
    permits_dfs.append(temp_df_q)

# Combine all regional permit dataframes into one master frame
if permits_dfs:
    df_permits_master = pd.concat(permits_dfs, ignore_index=True)
    print(f"Successfully processed {len(permits_dfs)} building permit regional files!")
    display(df_permits_master.head())
else:
    print("No building permit CSV files found in the folder. Check directory path.")

Processing Building Permits files from folder...
Successfully processed 6 building permit regional files!


,year,quarter,building_permits,metro_name
0,1988,1,2143,"Kansas City, MO-KS"
1,1988,2,3434,"Kansas City, MO-KS"
2,1988,3,2789,"Kansas City, MO-KS"
3,1988,4,2554,"Kansas City, MO-KS"
4,1989,1,1762,"Kansas City, MO-KS"


In [9]:
# 6. MASTER MERGE
print("Merging datasets into master modeling frame...")

# Clean HPI dataset
df_master = df_hpi.copy()

# 1. Merge national Mortgage Rates on year & quarter
df_master = pd.merge(df_master, df_mortgage_q, on=['year', 'quarter'], how='left')

# 2. Merge statewide Median Income on year
df_master = pd.merge(df_master, df_income, on=['year'], how='left')

# 3. Merge regional Unemployment on metro name, year, and quarter
if 'df_unemp_master' in locals() and not df_unemp_master.empty:
    df_master = pd.merge(df_master, df_unemp_master, on=['metro_name', 'year', 'quarter'], how='left')

# 4. Merge regional Building Permits on metro name, year, and quarter
if 'df_permits_master' in locals() and not df_permits_master.empty:
    df_master = pd.merge(df_master, df_permits_master, on=['metro_name', 'year', 'quarter'], how='left')

# Save final merged modeling dataset
os.makedirs('../data/processed', exist_ok=True)
df_master.to_csv('../data/processed/kansas_housing_model_master.csv', index=False)

print(f"Master dataset saved successfully! Final shape: {df_master.shape}")
display(df_master.head(15))

Merging datasets into master modeling frame...
Master dataset saved successfully! Final shape: (912, 11)


,metro_name,msa_code,year,quarter,index_value,standard_error,quarterly_growth,mortgage_rate,median_income,unemployment_rate,building_permits
0,"Kansas City, MO-KS",28140,1976,2,45.25,( 0.54),0.689809,8.776923,NaN,NaN,NaN
1,"Kansas City, MO-KS",28140,1976,3,46.30,( 0.48),2.320442,8.967692,NaN,NaN,NaN
2,"Kansas City, MO-KS",28140,1976,4,46.19,( 0.50),-0.237581,8.847143,NaN,NaN,NaN
3,"Kansas City, MO-KS",28140,1977,1,47.15,( 0.57),2.078372,8.693333,NaN,NaN,NaN
4,"Kansas City, MO-KS",28140,1977,2,49.72,( 0.45),5.450689,8.807692,NaN,NaN,NaN
5,"Kansas City, MO-KS",28140,1977,3,51.61,( 0.40),3.801287,8.925000,NaN,NaN,NaN
6,"Kansas City, MO-KS",28140,1977,4,53.88,( 0.49),4.398372,8.936923,NaN,NaN,NaN
7,"Kansas City, MO-KS",28140,1978,1,55.66,( 0.55),3.303638,9.126923,NaN,NaN,NaN
8,"Kansas City, MO-KS",28140,1978,2,58.04,( 0.48),4.275961,9.557692,NaN,NaN,NaN
9,"Kansas City, MO-KS",28140,1978,3,59.78,( 0.48),2.997932,9.760000,NaN,NaN,NaN
